# Arms 6 and 7' (pilot): FracAtlas transfer, seed 42, 5 folds

* **Arm 6**: pre-train YOLOv8s on FracAtlas (one class: fracture, 50 epochs), then fine-tune on each HBFMID-Clean fold exactly like the baseline (100 epochs, last.pt). The FracAtlas checkpoint is trained once (seed 42), so pre-training variance is not part of the pilot spread.
* **Arm 7'**: joint training, HBFMID train + all FracAtlas fractured images labelled with an extra 11th class ("generic fracture"); val/test stay HBFMID-only; the generic class counts as a localiser in the recall metric.
* `VARIANT = "generic"` uses the original FracAtlas images; `"zoom"` uses scale-matched crops (FracAtlas boxes are ~2.4x smaller than HBFMID boxes; the crops bring them to ~88 px at imgsz 640). Run `generic` first, `zoom` as a separate Save Version.

FracAtlas and HBFMID share no image (cross-dataset audit), so nothing can leak into the test folds. 59 broken FracAtlas JPEGs are skipped.
Settings: Accelerator *GPU T4 x2*, Internet ON, **both datasets added** (hbfmid and fracatlasfigshare). Estimated 3-4 h. Before running: `results/baseline_fixed` pushed to GitHub (the comparison needs the baseline).

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215"], check=True)
os.environ["WANDB_MODE"] = "disabled"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
FA_ROOT = "/kaggle/input/datasets/nithoukhan/fracatlasfigshare/FracAtlas"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
assert os.path.isdir(FA_ROOT + "/images"), FA_ROOT
SMOKE = False
VARIANT = "generic"          # "generic" | "zoom"
MODEL = "yolov8s.pt"
SEED = 42
WORK = "/kaggle/working/fa_data"
if SMOKE:
    EPOCHS, PRE_EPOCHS, OUT, FOLDS, LIMIT = 2, 2, f"/kaggle/working/results/arms_fa_{VARIANT}_smoke", [0], 40
else:
    EPOCHS, PRE_EPOCHS, OUT, FOLDS, LIMIT = 100, 50, f"/kaggle/working/results/arms_fa_{VARIANT}", [0, 1, 2, 3, 4], 0
for cid in (0, 10):
    d = f"{WORK}/fa_{VARIANT}_c{cid}"
    if not os.path.exists(d + "/summary.json"):
        subprocess.run([sys.executable, f"{REPO}/scripts/data/prepare_fracatlas_yolo.py", "--fa-root", FA_ROOT,
                        "--manifest", f"{REPO}/data_splits/fracatlas/manifest.csv", "--out", d, "--variant", VARIANT,
                        "--cls-id", str(cid), "--limit", str(LIMIT)], check=True)
print(open(f"{WORK}/fa_{VARIANT}_c10/summary.json").read())

In [ ]:
PRE_TAG = f"fa_pre_{VARIANT}_s{SEED}"
PRE_CKPT = f"{OUT}/pretrain/{PRE_TAG}/weights_last.pt"

def pretrain_cmd(gpu):
    return [sys.executable, f"{REPO}/scripts/training/pretrain_fracatlas.py", "--data-dir", f"{WORK}/fa_{VARIANT}_c0", "--model", MODEL,
            "--epochs", str(PRE_EPOCHS), "--seed", str(SEED), "--device", str(gpu), "--out", f"{OUT}/pretrain", "--tag", PRE_TAG]

def arm_cmd(fold, arm):
    def f(gpu):
        c = [sys.executable, f"{REPO}/scripts/training/run_baseline.py", "--mode", "clean", "--fold", str(fold), "--seed", str(SEED),
             "--model", MODEL, "--epochs", str(EPOCHS), "--fixed-epochs", "--device", str(gpu), "--data-root", DATA_ROOT,
             "--splits-dir", f"{REPO}/data_splits", "--out", OUT, "--arm", arm]
        if arm.startswith("fapre"):
            c += ["--init-weights", PRE_CKPT]
        else:
            c += ["--extra-train", f"{WORK}/fa_{VARIANT}_c10/lists/fractured_all.txt", "--extra-names", "FracAtlas fracture"]
        return c
    return f

ARM6, ARM7 = f"fapre_{VARIANT}", f"fajoint_{VARIANT}"
# order: pre-training first (one GPU), joint-training folds meanwhile on the other GPU, arm 6 folds wait for the checkpoint
TASKS = [(PRE_TAG, pretrain_cmd, None)]
TASKS += [(f"clean_f{f}_s{SEED}_{ARM7}", arm_cmd(f, ARM7), None) for f in FOLDS]
TASKS += [(f"clean_f{f}_s{SEED}_{ARM6}", arm_cmd(f, ARM6), PRE_CKPT) for f in FOLDS]
print(len(TASKS), "jobs, epochs =", EPOCHS, "->", OUT)

In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count()))
q = queue.Queue()
for t in TASKS:
    q.put(t)
failed = []

def worker(gpu):
    while True:
        try:
            name, cmd_fn, needs = q.get_nowait()
        except queue.Empty:
            return
        while needs and not os.path.exists(needs):          # e.g. arm 6 waits for the FracAtlas pre-trained checkpoint
            if any(n.startswith("fa_pre_") for n in failed):
                break
            time.sleep(20)
        t0 = time.time()
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd_fn(gpu), stdout=f, stderr=subprocess.STDOUT).returncode
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time()-t0)/60:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none")
for n in failed:
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-25:]))

In [ ]:
import shutil
BASE = [f"{REPO}/results/controls", f"{REPO}/results/baseline_fixed"]
for arm in (ARM6, ARM7):
    subprocess.run([sys.executable, f"{REPO}/scripts/evaluation/aggregate_arm.py", "--name", f"arm_{arm}", "--runs", OUT, "--arm-tag", arm, "--out", OUT])
    subprocess.run([sys.executable, f"{REPO}/scripts/evaluation/compare_arms.py", "--data-root", DATA_ROOT,
                    "--baseline-runs", *BASE, "--arm-runs", OUT, "--arm-name", arm, "--arm-tag", arm, "--out", f"{OUT}/{arm}_vs_baseline"])
shutil.copy(f"{WORK}/fa_{VARIANT}_c10/summary.json", f"{OUT}/fa_prep_summary_{VARIANT}.json")
shutil.make_archive(f"/kaggle/working/arms_fa_{VARIANT}_results", "zip", OUT)      # the 22 MB pre-trained checkpoint is inside; delete it from the folder before git add
print(sorted(os.listdir(OUT)))
if failed:
    print("\nWARNING: failed runs:", failed)

After the run: download `arms_fa_<variant>_results.zip`, unzip into `results/arms_fa_<variant>/`, **delete `pretrain/*/weights_last.pt`** (22 MB, not for git), commit and push.